**Vanguard**  
**Jaydon Hylton (2210144)** , **Chadwick Cox (1800729)** , **Semoy Smith (1505625)**

# Part 2: Dynamic Persona Prompting and Interactive Chat Models

`SystemMessage` sets advising policy. `HumanMessage` carries the student inquiry. `ChatPromptTemplate.from_messages()` swaps persona and query at runtime. `textwrap.dedent` / `textwrap.fill` format console output.

In [ ]:
%pip install -q azure-identity langchain-openai langchain-core python-dotenv

In [ ]:

from __future__ import annotations

import os
from pathlib import Path

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

load_dotenv(Path.cwd() / ".env")
try:
    from google.colab import userdata

    for name in (
        "AZURE_AI_FOUNDRY_ENDPOINT",
        "AZURE_AI_FOUNDRY_KEY",
        "AZURE_MODEL_NAME",
        "AZURE_OPENAI_API_VERSION",
    ):
        if not os.getenv(name):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass
except ImportError:
    pass


def build_chat_model() -> AzureChatOpenAI:
    endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT", "").strip().rstrip("/")
    kwargs = {
        "azure_endpoint": endpoint,
        "azure_deployment": os.getenv("AZURE_MODEL_NAME", "gpt-4.1-mini").strip(),
        "api_version": os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21").strip(),
        "temperature": 0.0,
        "max_tokens": 256,
    }
    if os.getenv("AZURE_AUTH_MODE", "local").lower() == "identity":
        kwargs["azure_ad_token_provider"] = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )
    else:
        kwargs["api_key"] = os.getenv("AZURE_AI_FOUNDRY_KEY")
    return AzureChatOpenAI(**kwargs)

## Role-based messages and console formatting

In [ ]:
import textwrap

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

SENIOR_ADVISOR = textwrap.dedent(
    """
    You are a Senior Computer Science Academic Advisor at the University of Technology.
    Advise only on prerequisites, GPA, registration holds, and degree requirements.
    Refuse legal, medical, or non-academic requests. Cite course codes. Keep answers short.
    """
).strip()

REGISTRAR_AUDITOR = textwrap.dedent(
    """
    You are a Faculty Registrar Auditor. Reply in numbered policy findings only.
    Do not give career advice. Flag missing transcript evidence.
    """
).strip()

system = SystemMessage(content=SENIOR_ADVISOR)
human = HumanMessage(
    content="Student ID: 816001234\n\nQuestion: What are the prerequisites for COMP 3901?"
)
print("=== System ===")
print(textwrap.fill(system.content, width=88))
print("\n=== Human ===")
print(textwrap.fill(human.content, width=88))

## Reusable `ChatPromptTemplate.from_messages()` with runtime persona changes

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "{persona}"),
        ("human", "Student ID: {student_id}\n\nQuestion: {student_question}"),
    ]
)
chain = prompt | build_chat_model()
query = {
    "student_id": "816001234",
    "student_question": "Summarize the prerequisites for COMP 3901.",
}

senior = chain.invoke({**query, "persona": SENIOR_ADVISOR})
print("=== Senior Computer Science Academic Advisor ===")
print(textwrap.fill(str(senior.content), width=88))

auditor = chain.invoke({**query, "persona": REGISTRAR_AUDITOR})
print("\n=== Faculty Registrar Auditor ===")
print(textwrap.fill(str(auditor.content), width=88))